In [201]:
import pandas as pd

df = pd.read_csv('chosen-images-mini.txt')

images = list(df["image"])
languages = list(df["language"])
    
images

['ghcr.io/patcheval-cve/patcheval-cve:cve-2015-8213',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2017-1000189',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2017-16042',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2018-20834',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2019-10787',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2020-15233',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2020-15278',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2020-26226',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2020-26299',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2020-4037',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2020-7764',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2021-21360',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2021-21384',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2021-22538',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2021-23727',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2021-32701',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2021-32796',
 'ghcr.io/patcheval-cve/patcheval-cve:cve-2021-32803',
 'ghcr.io/p

In [202]:
languages

['Python',
 'JavaScript',
 'JavaScript',
 'JavaScript',
 'JavaScript',
 'Go',
 'Python',
 'JavaScript',
 'JavaScript',
 'Go',
 'JavaScript',
 'Python',
 'JavaScript',
 'Go',
 'Python',
 'Go',
 'JavaScript',
 'JavaScript',
 'Python',
 'JavaScript',
 'Python',
 'Python',
 'JavaScript',
 'JavaScript',
 'Python',
 'Python',
 'JavaScript',
 'JavaScript',
 'JavaScript',
 'Go',
 'JavaScript',
 'Python',
 'Go',
 'Go',
 'JavaScript',
 'JavaScript',
 'Go',
 'Go',
 'Go',
 'Go',
 'Go',
 'Python',
 'Go',
 'Python',
 'Go',
 'Python',
 'Python',
 'Python',
 'Go',
 'Python',
 'Python',
 'Python',
 'Python',
 'Go',
 'Go',
 'Go',
 'JavaScript',
 'Go',
 'Go',
 'Python']

In [203]:
def get_cve (image_name):
    cve = image_name.split(':')[-1].upper()
    return cve

get_cve(images[0])

'CVE-2015-8213'

In [204]:
from pathlib import Path

TASK_DIR = Path('tasks') / "58008-60"

def get_docker_path (cve):
    return TASK_DIR / cve / 'environment'

def get_task_folder(cve):
    return TASK_DIR / cve

get_docker_path(get_cve(images[0]))

PosixPath('tasks/58008-60/CVE-2015-8213/environment')

In [205]:
Docker_file_template = """
FROM {}
RUN mv /workspace /app
WORKDIR /app

RUN find . -mindepth 1 -maxdepth 1 \\
    ! -exec sh -c '[ -d "$1" ] && [ -d "$1/.git" ]' _ {{}} \\; \\
    -exec rm -rf -- {{}} +

RUN apt-get update \\
    && apt-get install -y --no-install-recommends \\
        bash curl nodejs npm procps ripgrep \\
    && rm -rf /var/lib/apt/lists/*

ENTRYPOINT ["/bin/bash", "-c", "exec tail -f /dev/null"]
"""

import os

for image in images:
    cve = get_cve(image)
    env_folder = get_docker_path(cve)
    os.makedirs(env_folder, exist_ok=True)
    dockerfile = env_folder / "Dockerfile"
    with open(dockerfile, "w") as f:
        f.write(Docker_file_template.format(image).strip())   
    

In [ ]:
Docker_file_template = """
FROM {}
WORKDIR /workspace
COPY . /tests/
ENTRYPOINT ["/bin/bash", "-c", "exec tail -f /dev/null"]
"""

test_sh_template = """ 
#!/bin/bash
set -e

# Run PatchEval validation from /workspace.
# Do not run prepare.sh, as it may mutate lockfiles or module metadata.
echo "[*] Initializing..."
cp -a /app/. /workspace/ || {
  echo "[*] Failed to initialize!"
  exit 1
}

command -v uvx >/dev/null 2>&1 || curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"
command -v uvx >/dev/null 2>&1 || { echo "[*] Failed to locate uv!"; exit 1; }

cd /workspace && rm -rf /app

echo "[*] Generating patches..."
bash /tests/gen-patch.sh

echo "[*] Running tests..."
uvx \\
  --python '>=3.12,<3.14' \\
  --from 'harbor-rewardkit==0.2.*' \\
  --with-requirements /tests/requirements.txt \\
  rewardkit /tests \\
  --workspace /workspace \\
  --output /logs/verifier/reward.json

echo "[+] All tests completed!"
"""

# CVE-2023-33977 laf kiwi not Kiwi
gen_patch_template = """ 
#!/bin/bash
set -e

cd /workspace/{}

if ! git rev-parse --is-inside-work-tree >/dev/null 2>&1; then
  echo "[*] Target workdir is not a git repository: $(pwd)" >&2
  exit 1
fi

echo "[*] Git diff..."
git diff HEAD -U3 > /logs/verifier/git.patch

for f in /workspace/fix.patch /workspace/llm.patch; do
    if [ -f "$f" ]; then
        cp "$f" "/logs/verifier/"
        echo "Copied $f -> /logs/verifier/"
    else
        echo "Skipped $f (not found)"
    fi
done

# If git.patch is empty, fall back to llm.patch when it exists
if [ ! -s /logs/verifier/git.patch ] && [ -f /workspace/llm.patch ]; then
    cp /workspace/llm.patch /logs/verifier/git.patch
fi

cp /logs/verifier/git.patch /workspace/fix.patch

echo "[*] All patches completed!"
"""

import os

for image in images:
    cve = get_cve(image)
    env_folder = get_task_folder(cve) / "tests"
    os.makedirs(env_folder, exist_ok=True)
    dockerfile = env_folder / "Dockerfile"
    testfile = env_folder / "test.sh"
    patchfile = env_folder / "gen-patch.sh"
    with open(dockerfile, "w") as f:
        f.write(Docker_file_template.format(image).strip())   
    
    with open(testfile, 'w') as f:
        f.write(test_sh_template.strip())
    
    with open(patchfile, "w") as f:
        f.write(gen_patch_template.format(get_folder_name(cve)).strip())

In [207]:
task_yaml_template = '''
schema_version = "1.4"
artifacts = [
    {{ source = "/app", exclude = [".git"] }},
]

[task]
name = "PatchEval/{}"
version = "1.0.0"
description = ""
authors = []
keywords = []

[metadata]

[verifier]
environment_mode = "separate"
collect = []

[verifier.environment]
cpus = 2
memory_mb = 1024

[verifier.env]

[agent]
network_mode = "allowlist"
allowed_hosts = [
    "openrouter.ai",
    "api.openai.com",
    "api.anthropic.com",
    "generativelanguage.googleapis.com",
    "api.deepseek.com",
    "api.mistral.ai",
    "api.groq.com",
    "api.together.xyz",
    "api.fireworks.ai",
    "api.cerebras.ai",
    "api.x.ai",
    "router.huggingface.co",
    "raw.githubusercontent.com",
]
timeout_sec = 1800

[environment]
network_mode = "public"
cpus = 1
memory_mb = 2048
storage_mb = 10240

[environment.env]

[solution.env]
'''

for image in images:
    cve = get_cve(image)
    env_folder = get_task_folder(cve)
    os.makedirs(env_folder, exist_ok=True)
    task_toml = env_folder / "task.toml"
    with open(task_toml, "w") as f:
        f.write(task_yaml_template.format(cve).strip())   

In [208]:
dataset_file = "/Users/nguyendangduong/Data/TITAN/agent/PatchEval/patcheval/datasets/patcheval_verified.json"
import json
from collections import defaultdict

with open(dataset_file, 'r') as f:
    content = json.load(f)

cve_map = defaultdict(dict)
for item in content:
    cve_id = item.get("cve_id", None)
    if cve_id:
        cve_map[cve_id] = item
        
cve_map['CVE-2021-23376']

{'cve_id': 'CVE-2021-23376',
 'cve_description': 'This affects all versions of package ffmpegdotjs. If attacker-controlled user input is given to the trimvideo function, it is possible for an attacker to execute arbitrary commands. This is due to use of the child_process exec function without input sanitization.',
 'cwe_info': {'CWE-78': {'name': "Improper Neutralization of Special Elements used in an OS Command ('OS Command Injection')",
   'description': 'The product constructs all or part of an OS command using externally-influenced input from an upstream component, but it does not neutralize or incorrectly neutralizes special elements that could modify the intended OS command when it is sent to a downstream component.'}},
 'repo': 'https://github.com/TRomesh/ffmpegdotjs',
 'patch_url': ['https://github.com/TRomesh/ffmpegdotjs/commit/dae868764d6418dac081ed1aa84c636645ba6adb'],
 'programming_language': 'JavaScript',
 'vul_func': [{'id': 'vul_js_21_1',
   'commit': 'b7395da',
   'file

In [209]:
def get_folder_name(cve):
    repo = cve_map[cve]['repo']
    return str(repo).split('/')[-1]

In [210]:
import re


def redact_cve_description(description: str | None) -> str:
    """Redact CVE-identifying metadata and source-code references."""

    if not description:
        return ""

    # Remove Markdown links, preserving neither link text nor URL.
    description = re.sub(
        r"!?\[([^\]]*)\]\(\s*<?(?:https?://|www\.)[^)]*>?[^)]*\)",
        "[REDACTED]",
        description,
        flags=re.IGNORECASE,
    )

    # Remove bare URLs, including escaped Markdown URLs.
    description = re.sub(
        r"(?:https?://|ftp://|www\.)[^\s<>`)\]]+",
        "[REDACTED]",
        description,
        flags=re.IGNORECASE,
    )

    # Remove CVE/advisory identifiers.
    description = re.sub(
        r"\b(?:CVE-\d{4}-\d{4,}|"
        r"GHSA-[A-Za-z0-9-]+|"
        r"CWE-\d+|"
        r"OSV-\d{4}-\d+|"
        r"ZDI-\d{2}-\d+)\b",
        "[REDACTED]",
        description,
        flags=re.IGNORECASE,
    )

    # Remove PR, pull request, issue, and review references.
    description = re.sub(
        r"\b(?:pull\s+request|PR|issue|commit|"
        r"merge\s+request|MR)\s*#\s*\d+\b",
        "[REDACTED]",
        description,
        flags=re.IGNORECASE,
    )

    # Remove GitHub-style references such as #424.
    description = re.sub(r"(?<!\w)#\d+\b", "[REDACTED]", description)

    # Remove Git commit hashes.
    description = re.sub(
        r"\b(?:[0-9a-f]{7,40})\b",
        "[REDACTED]",
        description,
        flags=re.IGNORECASE,
    )

    # Remove source-code paths, including paths inside backticks.
    description = re.sub(
        r"`[^`\n]*(?:\.go|\.py|\.js|\.ts|\.java|\.c|\.cc|\.cpp|"
        r"\.h|\.hpp|\.rs|\.php|\.rb|\.cs|\.swift|\.kt)"
        r"(?:#[A-Za-z0-9_.$-]+)?[^`\n]*`",
        "[REDACTED]",
        description,
        flags=re.IGNORECASE,
    )

    description = re.sub(
        r"(?<!\w)(?:[\w.-]+/)+[\w.-]+\."
        r"(?:go|py|js|ts|java|c|cc|cpp|h|hpp|rs|php|rb|cs|swift|kt)"
        r"(?:#L\d+(?:-L?\d+)?)?",
        "[REDACTED]",
        description,
        flags=re.IGNORECASE,
    )

    # Remove function and method references in backticks.
    # Examples:
    # `tokenFromCache()`
    # `AuthenticatorOAuth2Introspection.Authenticate(...)`
    # `func (a *AuthenticatorOAuth2Introspection) Authenticate(...)`
    description = re.sub(
        r"`[^`\n]*(?:"
        r"\bfunc\s*\([^)]*\)\s*[\w.]+\s*\([^`]*"
        r"|"
        r"\b[\w.$*]+\s*\([^`]*\)"
        r")[^`\n]*`",
        "[REDACTED]",
        description,
        flags=re.IGNORECASE,
    )

    # Remove standalone function/method call references.
    description = re.sub(
        r"\b(?:func\s*)?"
        r"(?:\([^)]{1,100}\)\s*)?"
        r"[A-Za-z_]\w*(?:\.[A-Za-z_]\w*)*"
        r"\s*\([^()\n]{0,150}\)",
        "[REDACTED]",
        description,
    )

    # Remove inline code identifiers, version strings, and code symbols.
    description = re.sub(
        r"`[^`\n]*`",
        "[REDACTED]",
        description,
    )

    # Remove release versions such as v0.38.12-beta.1.
    description = re.sub(
        r"\bv?\d+\.\d+(?:\.\d+)+(?:[-+][A-Za-z0-9.-]+)?\b",
        "[REDACTED]",
        description,
        flags=re.IGNORECASE,
    )

    # Remove explicit security/CVE terminology.
    description = re.sub(
        r"\b(?:CVE|CWE|CVSS|NVD)\b",
        "[REDACTED]",
        description,
        flags=re.IGNORECASE,
    )

    description = re.sub(
        r"\b(?:security\s+)?"
        r"(?:vulnerabilit(?:y|ies)|"
        r"security\s+(?:flaw|issue|advisory|bug)|"
        r"exploit(?:ation)?|"
        r"proof.of.concept|"
        r"responsible\s+disclosure)\b",
        "[REDACTED]",
        description,
        flags=re.IGNORECASE,
    )

    # Remove common explicit remediation/release disclosures.
    description = re.sub(
        r"\b(?:a\s+)?patch\s+will\s+be\s+released\b",
        "[REDACTED]",
        description,
        flags=re.IGNORECASE,
    )

    # Clean whitespace and punctuation left by redaction.
    description = re.sub(r"(?:\s*\[REDACTED\]\s*){2,}", " [REDACTED] ", description)
    description = re.sub(r"\s+", " ", description).strip()

    return description

In [211]:
def _prompt(sample: dict[str, Any], workdir_hint: str = "/app") -> str:
    return (
        "## USER\n\n"
        "Please fix the vulnerabilities in the code repository based on the following information: "
        + redact_cve_description(str(sample.get("cve_description", ""))).strip()
        + "\n\n"
        + "Task runtime information:\n"
        + "- Target workdir: "
        + workdir_hint
        + "\n"
        + "- All tool path arguments must stay under this directory.\n"
        + f"- Start exploration from this workspace root instead of guessing a path under {workdir_hint}.\n"
        + f"- Before stopping, write the final repository diff to `{workdir_hint}/llm.patch` from the target workdir."
        + "\n\n"
        + "Repair-source restrictions:\n"
        + "- Do not search the web for this vulnerability, CVE, advisory, GHSA, release note, issue, pull request, or upstream patch.\n"
        + "- Do not run network commands such as curl, wget, git fetch, git pull, git ls-remote, npm view, pip index, or package/advisory lookups to find the fix.\n"
    )
    
for image in images:
    cve = get_cve(image)
    instruction = _prompt(cve_map.get(cve, ""))
    
    env_folder = get_task_folder(cve)
    os.makedirs(env_folder, exist_ok=True)
    task_toml = env_folder / "instruction.md"
    with open(task_toml, "w") as f:
        f.write(instruction.strip())   
    

In [212]:
requirement_template = """ 
codebleu[all]==0.7.0
tree-sitter-python==0.21.0
tree-sitter-go==0.21.0
tree-sitter-javascript==0.21.0
"""

In [213]:
test_vul_run_template = """ 
import rewardkit as rk
from rewardkit import criterion
from pathlib import Path

import json
import subprocess

@criterion
def test_vul_run(workspace: Path) -> dict:
    result = subprocess.run(
        ["bash", "/workspace/vul-run.sh"],
        capture_output=True,
        text=True,
    )

    stdout = result.stdout
    stderr = result.stderr
    exit_code = result.returncode
    return {
        "score": exit_code == 0,
        "reasoning": stdout
    }
"""

In [214]:
sim_template = ''' 
import json
import re
from pathlib import Path

import rewardkit as rk
from rewardkit import criterion

from codebleu import calc_codebleu

REFERENCE_PATCH = Path("/logs/verifier/fix.patch")
PREDICTION_PATCH = Path("/logs/verifier/git.patch")
WEIGHTS = (0.25, 0.25, 0.25, 0.25)  # ngram, weighted ngram, syntax, dataflow

EXTENSION_TO_LANGUAGE = {
    ".js": "javascript",
    ".jsx": "javascript",
    ".mjs": "javascript",
    ".cjs": "javascript",
    ".go": "go",
    ".py": "python",
}

DIFF_HEADER_RE = re.compile(r"diff --git a/.*? b/(.*)$")


def parse_added_lines(patch_text: str) -> dict[str, list[str]]:
    """Return {file path: added lines} for a (multi-file) git patch.

    Everything that is not an added line inside a hunk is dropped:
    `diff --git`, `index`, `---`, `+++`, `@@`, context/removed lines and
    `\\\\ No newline at end of file` markers.
    """
    files: dict[str, list[str]] = {}
    path, in_hunk = None, False

    for line in patch_text.splitlines():
        if (m := DIFF_HEADER_RE.match(line)):
            path, in_hunk = m.group(1), False
            files.setdefault(path, [])
        elif line.startswith("@@"):
            in_hunk = True
        elif in_hunk and path is not None and line.startswith("+"):
            files[path].append(line[1:])

    return files


def language_of(path: str) -> str | None:
    return EXTENSION_TO_LANGUAGE.get(Path(path).suffix.lower())


def detect_language(files: dict[str, list[str]]) -> str:
    """Single CodeBLEU language of the files that have added lines."""
    languages = {language_of(p) for p, added in files.items() if added} - {None}

    if not languages:
        raise ValueError("Could not detect a supported language in the patch")
    if len(languages) > 1:
        raise ValueError(f"Patch contains multiple languages: {sorted(languages)}")
    return languages.pop()


def added_code(files: dict[str, list[str]], language: str) -> str:
    """Added code of all files in `language`, in sorted path order."""
    return "\\n".join(
        line
        for path in sorted(files)
        if language_of(path) == language
        for line in files[path]
    )


def calculate_codebleu(
    reference_patch: Path,
    prediction_patch: Path,
    weights: tuple[float, float, float, float] = WEIGHTS,
) -> dict:
    """CodeBLEU between the added code of two git patches."""
    ref_files = parse_added_lines(Path(reference_patch).read_text(encoding="utf-8"))
    pred_files = parse_added_lines(Path(prediction_patch).read_text(encoding="utf-8"))

    language = detect_language(ref_files)
    reference = added_code(ref_files, language)
    prediction = added_code(pred_files, language)

    if not prediction.strip():  # agent added no code -> nothing to score
        return {"codebleu": 0.0, "language": language, "error": "empty prediction"}

    result = calc_codebleu(
        references=[reference],
        predictions=[prediction],
        lang=language,
        weights=weights,
    )
    return {**result, "language": language}


@criterion(description="CodeBLEU")
def CodeBLEU(workspace: Path) -> dict:
    result = calculate_codebleu(REFERENCE_PATCH, PREDICTION_PATCH)
    return {"score": result["codebleu"], "reasoning": json.dumps(result)}
'''

In [215]:
diff_template = ''' 
import json
import re
from pathlib import Path

import codebleu
import rewardkit as rk
from rewardkit import criterion

from codebleu import bleu, dataflow_match, syntax_match, weighted_ngram_match
from codebleu.utils import get_tree_sitter_language

REFERENCE_PATCH = Path("/logs/verifier/fix.patch")
PREDICTION_PATCH = Path("/logs/verifier/git.patch")
WEIGHTS = (0.3, 0.3, 0.2, 0.2)  # BLEU, BLEUw, Match_ast, Match_df

EXTENSION_TO_LANGUAGE = {
    ".js": "javascript",
    ".jsx": "javascript",
    ".mjs": "javascript",
    ".cjs": "javascript",
    ".go": "go",
    ".py": "python",
}

DIFF_HEADER_RE = re.compile(r"diff --git a/.*? b/(.*)$")

TOKEN_RE = re.compile(r"""
    (?P<comment>\\#[^\\n]*|//[^\\n]*|/\\*.*?\\*/)
  | (?P<string>[rbfRBF]{0,2}("{3}.*?"{3}|'{3}.*?'{3}|"(?:\\\\.|[^"\\\\])*"|'(?:\\\\.|[^'\\\\])*'|`[^`]*`))
  | (?P<number>\\b0[xX][0-9a-fA-F]+\\b|\\b\\d+(?:\\.\\d*)?(?:[eE][+-]?\\d+)?\\b)
  | (?P<ident>[A-Za-z_]\\w*)
  | (?P<op>@@|==|!=|<=|>=|->|<-|:=|\\+\\+|--|\\+=|-=|\\*=|/=|//|\\*\\*|&&|\\|\\||<<|>>|::|[^\\s\\w])
""", re.X | re.S)


# ------------------------------------------------------------ patch parsing
def parse_patch(patch_text: str) -> dict[str, list[str]]:
    """Return {file path: hunk lines} for a (multi-file) git patch.

    File headers (`diff --git`, `index`, `---`, `+++`) and
    `\\\\ No newline at end of file` markers are dropped. Hunk headers (`@@`)
    and the '+', '-' and ' ' lines inside hunks are kept (prefix preserved).
    """
    files: dict[str, list[str]] = {}
    path, in_hunk = None, False

    for line in patch_text.splitlines():
        if (m := DIFF_HEADER_RE.match(line)):
            path, in_hunk = m.group(1), False
            files.setdefault(path, [])
        elif line.startswith("@@") and path is not None:
            in_hunk = True
            files[path].append(line)
        elif in_hunk and path is not None and line[:1] in ("+", "-", " "):
            files[path].append(line)

    return files


def added_lines(files: dict[str, list[str]]) -> dict[str, list[str]]:
    return {p: [l[1:] for l in lines if l.startswith("+")] for p, lines in files.items()}


# ---------------------------------------------------------------- language
# Same logic as codebleu_criterion.py
def language_of(path: str) -> str | None:
    return EXTENSION_TO_LANGUAGE.get(Path(path).suffix.lower())


def detect_language(files: dict[str, list[str]]) -> str:
    """Single CodeBLEU language of the files that have added lines."""
    languages = {language_of(p) for p, added in files.items() if added} - {None}

    if not languages:
        raise ValueError("Could not detect a supported language in the patch")
    if len(languages) > 1:
        raise ValueError(f"Patch contains multiple languages: {sorted(languages)}")
    return languages.pop()


def only_language(files: dict[str, list[str]], language: str) -> dict[str, list[str]]:
    return {p: lines for p, lines in files.items() if language_of(p) == language}


# --------------------------------------------------------- diff-aware tokens
def tokenize_code(code: str) -> list[str]:
    """Drop comments, anonymize literals, keep keywords/operators."""
    return [
        {"string": "<STR>", "number": "<NUM>"}.get(m.lastgroup, m.group())
        for m in TOKEN_RE.finditer(code)
        if m.lastgroup != "comment"
    ]


def diff_tokenize(files: dict[str, list[str]]) -> list[str]:
    """Added tokens -> '+tok', removed tokens -> '-tok' (separate spaces).
    `@@` hunk-header tokens are kept unprefixed (line numbers become <NUM>).
    Files are visited in sorted-path order."""
    tokens = []
    for path in sorted(files):
        for line in files[path]:
            if line.startswith("@@"):
                tokens += tokenize_code(line)
            elif line[0] in "+-":
                tokens += [line[0] + t for t in tokenize_code(line[1:])]
    return tokens


def hunk_context(lines: list[str]) -> str:
    """Post-patch code of a file's hunks (context + added lines; `@@` lines excluded)."""
    return "\\n".join(l[1:] for l in lines if l[0] in "+ ")


def load_keywords(lang: str) -> set[str]:
    path = Path(codebleu.__file__).parent / "keywords" / f"{lang}.txt"
    return {w.strip() for w in path.read_text().splitlines()}


# ------------------------------------------------------------------ scoring
def calculate_diffbleu(
    reference_patch: Path,
    prediction_patch: Path,
    weights: tuple[float, float, float, float] = WEIGHTS,
) -> dict:
    """DiffBLEU = a*BLEU + b*BLEUw + c*Match_ast + d*Match_df between two git patches."""
    ref_all = parse_patch(Path(reference_patch).read_text(encoding="utf-8"))
    pred_all = parse_patch(Path(prediction_patch).read_text(encoding="utf-8"))

    language = detect_language(added_lines(ref_all))
    ref = only_language(ref_all, language)
    pred = only_language(pred_all, language)

    if not any(l.startswith(("+", "-")) for lines in pred.values() for l in lines):
        return {"diffbleu": 0.0, "language": language, "error": "empty prediction"}

    a, b, g, d = weights
    keywords = load_keywords(language)
    ref_tokens, pred_tokens = diff_tokenize(ref), diff_tokenize(pred)

    # BLEU / BLEUw over diff-aware tokens (the +/- prefix is ignored for keyword lookup)
    ref_weighted = [[ref_tokens, {t: 1 if t.lstrip("+-") in keywords else 0.2 for t in ref_tokens}]]
    s_bleu = bleu.corpus_bleu([[ref_tokens]], [pred_tokens])
    s_bleuw = weighted_ngram_match.corpus_bleu([ref_weighted], [pred_tokens])

    # Match_ast / Match_df: one instance per reference file, candidate = same path
    ref_ctx = [[hunk_context(lines)] for lines in ref.values()]
    pred_ctx = [hunk_context(pred.get(path, [])) for path in ref]
    tree_sitter_language = get_tree_sitter_language(language)
    s_ast = syntax_match.corpus_syntax_match(
        ref_ctx, pred_ctx, language, tree_sitter_language=tree_sitter_language
    )
    s_df = dataflow_match.corpus_dataflow_match(
        ref_ctx, pred_ctx, language, tree_sitter_language=tree_sitter_language
    )

    return {
        "diffbleu": a * s_bleu + b * s_bleuw + g * s_ast + d * s_df,
        "bleu": s_bleu,
        "bleu_w": s_bleuw,
        "match_ast": s_ast,
        "match_df": s_df,
        "language": language,
    }


@criterion(description="DiffBLEU")
def DiffBLEU(workspace: Path) -> dict:
    result = calculate_diffbleu(REFERENCE_PATCH, PREDICTION_PATCH)
    return {"score": result["diffbleu"], "reasoning": json.dumps(result)}
'''

In [216]:
sim_reward = ''' 
[[reward]]
weights = { CodeBLEU = 0.3, DiffBLEU = 0.7 }
'''

In [217]:
for image in images:
    cve = get_cve(image)
    task_folder = get_task_folder(cve)
    test_folder = task_folder / "tests"
    poc_folder = test_folder / "poc"
    sim_folder = test_folder / "similarity"
    
    os.makedirs(test_folder, exist_ok=True)
    os.makedirs(poc_folder, exist_ok=True)
    os.makedirs(sim_folder, exist_ok=True)
    
    requirement_txt = test_folder / "requirements.txt"
    with open(requirement_txt, "w") as f:
        f.write(requirement_template.strip())
        
    vul_run = poc_folder / "test_vul_run.py"
    with open(vul_run, "w") as f:
        f.write(test_vul_run_template.strip())
        
    sim_py = sim_folder / "CodeBLEU.py"
    with open(sim_py, "w") as f:
        f.write(sim_template.strip())
        
    diff_py = sim_folder / "DiffBLEU.py"
    with open(diff_py, "w") as f:
        f.write(diff_template.strip())
        
    sim_re = sim_folder / "reward.toml"
    with open(sim_re, "w") as f:
        f.write(sim_reward.strip())
    